# 22 - Reading the arithmetic contract

**Purpose.** To explain what notebook `21` established about PixInsight's subtraction and its
integration, and what a reader should now believe about each before contract 2 or contract 3 quotes
a number. `21` is the notebook that talked to the engine and made these numbers, and is written for
someone *checking* it. This one is written for someone *deciding what to do next* - specifically,
whether a pair difference or a stack taken through PixInsight can be trusted, and under exactly
which settings.

**What it is not for.** It measures nothing and writes nothing. Every number is read back from
`results/` - `pi_arithmetic_constants.json`, `pi_arithmetic_arms.csv`, `pi_arithmetic_ladder.csv`.
Where arithmetic appears below it is done on published numbers, to show what a published number is
worth; if any of it disagreed with `results/`, `results/` would be right and this notebook would be
the bug.

**It assumes `20_pi_contract_read.ipynb`** for the harness, the unit boundary at 65535, and why
PixInsight's MRS rather than our `1.4826 x MAD` is the right instrument wherever a spread is a few
counts. It does not repeat any of that.

**The headline: the engine does not lie, but four of its defaults do.**

1. **The subtraction is exact under the published settings**, and the *pedestal* is what makes it
   so - not the 32-bit float, which L23 prescribed in the same breath and which turns out to matter
   only for a difference thirty times wider than this camera can produce.
2. **The clip costs 0.583 of the truth, not half**, and the factor is a constant - so a clipped
   difference can never be recognised from its value.
3. **Averaging loses nothing.** With rejection off the engine reaches the ideal `sqrt(N)` at every
   rung, which settles that `eta_comb` is a number about the *combination* and never about the
   arithmetic.
4. **Four defaults are wrong for this work** - truncation on, format `SameAsTarget`, PSF-signal
   weighting, and a cache that outlives a settings change - and three of the four fail silently.

**Those are recomputed in section 1 rather than trusted from here.** A headline typed into markdown
goes stale the moment the notebook is re-run; the cell under section 1 asks the same questions of
the published file and will contradict this list if it ever stops being true.

In [ ]:
import json
import pathlib
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, str(pathlib.Path.cwd().parent))
from astropix import pixinsight as PI

RESULTS = pathlib.Path.cwd().parent / "results"
K = json.loads((RESULTS / "pi_arithmetic_constants.json").read_text(encoding="utf-8"))
arms = pd.read_csv(RESULTS / "pi_arithmetic_arms.csv")
ladder = pd.read_csv(RESULTS / "pi_arithmetic_ladder.csv")


def val(name):
    return K[name]["value"]


core = val("referee")
print(f"notebook 21 measured against {core['application']} {core['version']} "
      f"build {core['build']}")
print(f"  {len(K)} constants, {len(arms)} difference arms, {len(ladder)} integrations")
print(f"  on {val('synthetic_frames')['n_stack']} synthetic frames plus two pairs, "
      f"seed {val('synthetic_frames')['seed']}")

## 1. The four claims, answered from the file

Read this cell and you have the session. Everything below it is the working.

In [ ]:
ped = val("difference_settings")["pedestal"]
with_ped = arms[arms.pedestal == ped]
# Claim 1 is about the spread a bias pair can have.  The wide pair is thirty
# times that and is section 3's business, not the headline's.
with_ped_narrow = with_ped[with_ped.pair == "narrow"]
narrow_no_ped = arms[(arms.pedestal == 0.0) & (arms.pair == "narrow")]
clean = ladder[ladder.rejection == "none"]

c1 = with_ped_narrow.err_pct.abs().max() < 0.01
c2 = abs(val("clip_factor") - np.sqrt(0.5 - 1 / (2 * np.pi))) < 0.01
c3 = (clean.eta_comb - 1.0).abs().max() < 0.01
# "Quietly" means wrong by enough to matter and little enough to pass for a
# measurement.  An arm that is exact is not a failure at all.
quiet = int(((narrow_no_ped.err_pct < -10) & (narrow_no_ped.err_pct > -90)).sum())

print(f"1. the subtraction is exact with the pedestal, under all "
      f"{len(with_ped_narrow)} settings, at a bias pair's spread")
print(f"     worst error {with_ped_narrow.err_pct.abs().max():.6f}%   -> {c1}")
print(f"2. the clip factor is the half-normal, not a half")
print(f"     published {val('clip_factor'):.6f}, predicted "
      f"{np.sqrt(0.5 - 1 / (2 * np.pi)):.6f}   -> {c2}")
print(f"3. averaging reaches the ideal sqrt(N) at every rung")
print(f"     worst departure {100 * (clean.eta_comb - 1.0).abs().max():.2f}%   -> {c3}")
print(f"4. of the arms that fail, {quiet} fail quietly and "
      f"{int((narrow_no_ped.err_pct > 100).sum())} loudly")
assert c1 and c2 and c3, "the file disagrees with this notebook's headline; the file is right"

## 2. Why none of this could have been learned from a real frame

Every number in `21` comes from frames that were generated rather than captured, and that is the
methodological point rather than a convenience.

**Each failure below produces a plausible number.** A subtraction that clips reports a read noise
42% low - which, on a real bias pair, is indistinguishable from a quieter camera. A stack that
loses 3% to its rejection settings looks like a stack. An integration that refuses to run at all
returns `false` and writes nothing anywhere the caller can read. There is no feature of a real
frame that reveals any of them, because the thing that would reveal them is *the answer*, and on a
real frame nobody has it.

L23 said this in one line - do not check this on a real bias pair, inject a known sigma - and it is
the most useful sentence the retired projects left behind. It generalises past its own claim: **a
tool is checked against a known answer or it is not checked.**

**The frames are still written the way the camera writes.** Values are exact multiples of 16 in
stored units, so the reader and the `% 16` test that licenses this project's whole unit convention
do the same work here as on a real frame. A synthetic frame that skipped that would be exercising a
code path the camera never uses, which is a different kind of not-checking.

## 3. L23 was right about the mechanism and wrong about the remedy

The inherited claim: subtracting two 16-bit unsigned images clips every negative difference,
halving the apparent read noise, and the fix is to subtract in 32-bit float about a `+0.5` pedestal
with `rescale` and `truncate` off.

It prescribed two things at once. The table below is the eight-way split that separates them.

In [ ]:
narrow = arms[arms.pair == "narrow"].copy()
narrow["outcome"] = np.where(
    narrow.err_pct.abs() < 0.01, "exact",
    np.where(narrow.err_pct > 100, "wrapped -- loudly wrong", "clipped -- quietly wrong"))
print(narrow[["pedestal", "format", "truncated", "sigma", "err_pct",
              "frac_below_zero", "frac_at_zero", "outcome"]]
      .to_string(index=False, float_format=lambda v: f"{v:.4f}"))

print(f"\nwith the pedestal:    "
      f"{int((with_ped_narrow.err_pct.abs() < 0.01).sum())} of {len(with_ped_narrow)} exact")
print(f"without the pedestal: "
      f"{int((narrow_no_ped.err_pct.abs() < 0.01).sum())} of {len(narrow_no_ped)} exact")

### The pedestal is the fix. The float is insurance.

Every failure in that table is a difference that was left centred on zero, and every success is one
that was moved off it. With the pedestal present, the 16-bit path is **exactly** as good as the
float - same digits, not merely close.

That is worth stating plainly because it changes what has to be got right. A pair difference that
forgets the pedestal is broken regardless of its output format; a pair difference that has the
pedestal and uses 16-bit is fine. The two halves of the remedy are not equally load-bearing, and
only one of them is the thing to check in a review.

**The float still earns its keep, and section 3 of `21` shows where.** The pedestal buys half of
PixInsight's range on each side of zero. A difference wide enough that three sigma exceeds that runs
off the bottom anyway, and then the format decides whether the overflow is held or lost. It takes a
spread thirty times a real bias pair's to get there - which is why the float is insurance rather
than the remedy, and why it is kept anyway: it costs nothing.

In [ ]:
wide = arms[(arms.pair == "wide") & (arms.pedestal == ped)]
sf = val("synthetic_frames")
print(f"the wide pair is sigma {sf['sigma_wide_adc']:.0f} counts/frame, "
      f"{sf['sigma_wide_adc'] / sf['sigma_narrow_adc']:.0f}x the narrow one:\n")
print(wide[["format", "truncated", "sigma", "err_pct", "frac_below_zero", "min"]]
      .to_string(index=False, float_format=lambda v: f"{v:.4f}"))
exact = wide[wide.err_pct.abs() < 0.01]
print(f"\nexact here: {len(exact)} of {len(wide)} -- "
      f"{', '.join(f'{r.format}/truncate={r.truncated}' for r in exact.itertuples())}")
print(f"the others lose {wide[wide.err_pct.abs() > 0.01].err_pct.abs().min():.2f}% to "
      f"{wide.err_pct.abs().max():.2f}%")

## 4. 0.583, and why the sigma is not the evidence

Clipping a zero-centred normal at zero does not just discard half the pixels. It piles them onto a
single value, and what survives is a half-normal whose standard deviation is

```
sqrt(1/2 - 1/(2*pi))  =  0.5838
```

of the truth. L23 said "halves"; 0.584 is what that means, and the gap between the two is the gap
between a remembered number and a derived one.

**The consequence is the part that matters.** The factor does not depend on the sigma, on the
pedestal or on the frame. A clipped difference is always 58% of whatever the right answer was - so
it cannot be recognised by looking at it. There is no value that is suspicious, no range that is
implausible, nothing to notice.

**Which is why `pair_diff.js` reports a pixel census and not just a spread.** How many pixels sit
*below* zero against how many sit *exactly on* it: the same pixels, meaning opposite things - kept,
or clipped. That is a fact about the arithmetic rather than about the data, and it is the only
thing in the result that changes categorically when the trap fires.

In [ ]:
predicted = float(np.sqrt(0.5 - 1 / (2 * np.pi)))
clipped = arms[(arms.pedestal == 0.0) & (arms.frac_at_zero > 0.4)].copy()
clipped["ratio"] = clipped.sigma / clipped.truth

print(f"predicted {predicted:.6f}, published {val('clip_factor'):.6f}\n")
print(clipped[["pair", "format", "injected_sigma", "ratio"]]
      .to_string(index=False, float_format=lambda v: f"{v:.6f}"))
print(f"\nacross a {clipped.injected_sigma.max() / clipped.injected_sigma.min():.0f}x range of "
      f"injected sigma the ratio moves "
      f"{100 * (clipped.ratio.max() / clipped.ratio.min() - 1):.2f}%")
print("\nthe census, which does change categorically:")
print(f"  clipped arms:  {clipped.frac_at_zero.min():.1%} to {clipped.frac_at_zero.max():.1%} "
      f"of pixels sitting on exactly 0.0")
intact = arms[(arms.pedestal == 0.0) & (arms.frac_below_zero > 0.4)]
print(f"  intact arms:   {intact.frac_below_zero.min():.1%} to "
      f"{intact.frac_below_zero.max():.1%} of pixels BELOW zero, none piled on it")

## 5. Three refusals, and the console that explained them

None of these is in any inherited claim. All three were found by running the thing, and two of them
cost launches.

**`ImageIntegration` refuses fewer than three source images.** By name, whatever the rejection
setting. So the **N=2 rung of a doubling ladder does not exist through this engine** - which is not
academic, because session 03's `eta_comb` ladder on darks has one. Any comparison between that
ladder and a PixInsight one starts at 3.

**Its default weighting refuses a frame without stars.** `weightMode` defaults to PSF Signal
Weight, which weights each frame by the signal of the stars detected in it. A bias, a dark, a flat
and a synthetic frame all have none, so the weight is zero and `executeGlobal()` returns a bare
`false`. This one cost three launches and every setting in the matrix, because the failure is
identical under every combination of rejection, normalisation and stack size - it was never about
any of them.

**`for...in` over a process prototype takes the core down.** An access violation, not a catchable
exception. Probe by name.

### The console is retrievable, and that is the finding that unblocked the rest

`pjsr/NOTES.md` has opened since contract 1 with *PixInsight has no console, and everything follows
from that*. The premise stands - console output reaches no caller - but **`console.endLog()` returns
the log as a string**, so `harness.jsh` now wraps every run and attaches the tail to the result on
both paths.

That is not a tidiness improvement. A process that declines to run writes its reason to the console
**and nowhere else**; without the log, `false` is the entire diagnosis. With it, the reason arrived
as one line: *Zero or insignificant PSF Signal Weight estimate*.

**The general lesson is about where a failure is legible.** Contract 1 established that every script
must report by file because a silent script is unbisectable. This session found the other half: a
script can report perfectly and still say nothing useful, if what failed was a *process* rather than
the script. The log is how the process gets a voice.

In [ ]:
print("what a run must now state rather than inherit, and what the default would have done:\n")
for name, default, why in [
        ("PixelMath.truncate", "true",
         "clips the negative half of a difference -- silently, 41.6% low"),
        ("PixelMath.newImageSampleFormat", "SameAsTarget",
         "16-bit out of 16-bit in, so a negative wraps rather than being held"),
        ("ImageIntegration.weightMode", "PSF Signal Weight",
         "refuses any frame without stars, with a bare false"),
        ("ImageIntegration.useCache", "true",
         "a ladder re-runs the same paths under new settings, which is what a cache gets wrong")]:
    print(f"  {name:32s} defaults to {default:18s} {why}")

print(f"\npublished as constants, so a later run cannot rediscover them by accident:")
print(f"  integration_minimum_frames = {val('integration_minimum_frames')}")
print(f"  integration_weight_mode    = {val('integration_weight_mode')!r}")
print(f"  difference_settings        = {json.dumps(val('difference_settings'))}")

## 6. Averaging loses nothing, so `eta_comb` is a number about the rejection

The ladder is thirty-two frames of known noise, integrated at five stack sizes, twice: rejection
off, and Winsorized sigma clipping at 4.0/3.0.

In [ ]:
wide_table = ladder.pivot(index="n", columns="rejection", values="eta_comb")
wide_table["cost_pct"] = 100 * (1 - wide_table.winsorized)
print(wide_table.to_string(float_format=lambda v: f"{v:.4f}"))

rej = ladder[ladder.rejection == "winsorized"].sort_values("n")
print(f"\nrejection off: within {100 * (clean.eta_comb - 1).abs().max():.2f}% of the ideal at "
      f"every rung")
print(f"rejection on:  costs {100 * (1 - rej.eta_comb.iloc[0]):.2f}% at N={rej.n.iloc[0]}, "
      f"{100 * (1 - rej.eta_comb.iloc[-1]):.2f}% at N={rej.n.iloc[-1]} -- "
      f"a factor of {(1 - rej.eta_comb.iloc[0]) / (1 - rej.eta_comb.iloc[-1]):.1f}")

### What that settles

**`eta_comb` is not measuring the arithmetic.** The arithmetic is exact. Whatever a real `eta_comb`
falls short by is the *combination*: the rejection algorithm and its thresholds, the normalisation,
the weighting, and - once frames are registered - the resampling kernel. That is a useful thing to
know before the number is interpreted, because it means a low `eta_comb` is a statement about a
choice somebody made rather than about a tool's precision.

**The stack size is part of the constant, not context for it.** The same rejection setting costs
several times more at the bottom of the ladder than at the top. Rejection discards real pixels, and
at three frames there are too few survivors left to average well. MISSION already required
`eta_comb`'s provenance to record the stack size and the rejection settings; this is the measurement
that shows a number quoted without both is not a measurement of anything.

**These frames have no outliers**, so the rejection column is the pure cost of rejecting when there
is nothing to reject - a floor, not an estimate. On real frames rejection removes cosmic rays,
satellites and hot pixels, and buys back something this ladder cannot see. That difference is
contract 3's to measure, and it is why `eta_comb_engine` must never be used as `eta_comb`.

## 7. The one flaw in the ladder's own method, stated rather than hidden

`eta_comb_engine` comes out fractionally **above 1.0**, and it climbs with N - from about 1.000 at
N=3 to about 1.004 at N=32. An efficiency above 1 is not physical: it would be a stack quieter than
`sqrt(N)` allows.

The cause is in how the ladder measures its own denominator. `sigma_single` is taken from **one
frame** - and that frame is also *in* every stack. A frame whose own noise happens to sit high
inflates the ideal, and it inflates the stack too, but its share of the stack falls as `1/N`. So at
small N the two move together and at large N only the denominator is affected, which is exactly the
monotone climb observed.

**It is small enough not to matter and worth naming anyway.** The whole effect is 0.4%, against a
rejection cost of 1 to 7%, so nothing in section 6 turns on it. But it is a real correlation between
an estimate and the thing it is normalising, and a reader who found it unexplained would be right to
wonder what else was not checked.

**The fix, when a real `eta_comb` is measured:** take `sigma_single` as the mean over the frames in
the stack rather than from one of them, or from frames held out of it. Not done here because this
ladder is a calibration of the engine and 0.4% does not change its verdict - but contract 3's number
is the model's, and it should not inherit this.

In [ ]:
print(ladder[ladder.rejection == "none"][["n", "sigma_single", "sigma_stack", "ideal",
                                          "eta_comb"]]
      .to_string(index=False, float_format=lambda v: f"{v:.5f}"))
sf = val("synthetic_frames")
drift = clean.sort_values("n").eta_comb
print(f"\ninjected sigma was {sf['sigma_narrow_adc']:.1f}; the one frame used as the denominator "
      f"measured {clean.sigma_single.iloc[0]:.4f} "
      f"({100 * (clean.sigma_single.iloc[0] / sf['sigma_narrow_adc'] - 1):+.2f}%)")
print(f"eta_comb climbs {drift.iloc[0]:.5f} -> {drift.iloc[-1]:.5f} across the ladder, "
      f"{100 * (drift.iloc[-1] - drift.iloc[0]):.2f} points")
print("  a one-frame denominator that also sits inside every stack is the explanation, and the "
      "size is right")

## 8. What is settled, and what contract 2 and contract 3 inherit

The decision this notebook exists to support, spelled out.

In [ ]:
print("SETTLED")
print(f"  a pair difference through PixInsight returns the truth under "
      f"{json.dumps(val('difference_settings'))}")
print(f"  a clipped difference reads {val('clip_factor'):.4f} of the truth, always")
print(f"  averaging N frames loses nothing: eta_comb_engine is 1.0 to "
      f"{100 * (clean.eta_comb - 1).abs().max():.2f}% at every rung")
print(f"  ImageIntegration needs >= {val('integration_minimum_frames')} frames and an explicit "
      f"weight mode")
print()
print("CONTRACT 2 MAY NOW PROCEED -- g and R on a real bias pair through PixInsight.")
print("  It subtracts, and the subtraction is now checked.  Use the published settings; the")
print("  defaults are the failing configuration.")
print()
print("CONTRACT 3 MAY NOW PROCEED -- eta_comb on real, registered frames.")
print("  Its ladder starts at N=3, states its weighting, and must take sigma_single as a mean")
print("  over frames rather than from one of them (section 7).")
print()
print("NOT SETTLED, AND DELIBERATELY OUT OF SCOPE")
print("  Registration.  Nothing here resamples anything, so the resampling loss -- the gap")
print("  between session 03's dark ladder and a registered one -- is untouched.")
print("  What rejection BUYS.  These frames have no outliers, so the rejection column is a")
print("  floor on its cost and says nothing about its benefit.")
print("  Any property of the sensor.  This notebook measured the engine.  g, R, the pedestal,")
print("  F_sky, t_dead and session 03's eta_comb are all exactly where they were.")
print("  Any other build.  Every number above belongs to "
      f"{core['application']} {core['version']} build {core['build']}; re-run pjsr/probe.js")
print("  after an upgrade.  The half-normal factor is the exception -- it is arithmetic, and")
print("  tests/test_pixinsight.py asserts it in pure numpy with no PixInsight at all.")